In [ ]:
# =============================================================================
# DAY 26 — Multi-Agent Reinforcement Learning (MARL) Basics
# =============================================================================
# So far (Days 22–25): ONE agent in one environment (CartPole, Pendulum, LLM).
# Today: SEVERAL agents act at once. Their actions change the world together,
# and each may get its own reward (or share one).
#
# Everyday pictures:
#   Cooperative: two robots carrying a table — both must succeed together.
#   Competitive: chess / soccer opponents.
#   Mixed: self-driving cars — share the road but each wants its own destination.
#
# This notebook: concepts first, then three baselines on a tiny 2-agent Gridworld.
#
#
# ---------------------------------------------------------------------------
# 1) FORMAL SETTING — MARKOV GAMES (also called Stochastic Games)
# ---------------------------------------------------------------------------
#
# A Markov Game with N agents is the multi-agent generalization of an MDP.
#
# Ingredients:
#   S          = shared state space (what the world looks like)
#   A_i        = action set for agent i   (i = 1…N)
#   A          = A_1 × A_2 × … × A_N      joint action
#   P(s'|s, a) = transition given the JOINT action a = (a_1,…,a_N)
#   R_i(s, a)  = reward for agent i (may depend on everyone's actions)
#   γ          = discount
#   (optional) observation o_i = O_i(s) if agents don't see full state
#
# Special cases:
#   N = 1                         → ordinary MDP
#   all R_i equal (team reward)   → fully cooperative
#   R_1 = −R_2 (two players)      → zero-sum competitive
#   otherwise                     → general-sum / mixed
#
# Each agent i has a policy π_i(a_i | o_i) (or | s if fully observed).
# The joint policy is π = (π_1,…,π_N).
#
# Goal (cooperative view we use today):
#   maximize expected TEAM return  E[ Σ_t γ^t r_team(s_t, a_t) ]
#
# Important: from agent i's view, the "environment" includes the OTHER agents.
# If they change their policies while i learns, the world looks NON-STATIONARY.
# That is the root of most MARL pain.
#
#
# ---------------------------------------------------------------------------
# 2) THE CORE CHALLENGES OF MARL
# ---------------------------------------------------------------------------
#
# (A) Non-stationarity
#     Single-agent RL assumes a fixed P and R. In MARL, when teammate/opponent
#     updates π_j, the effective dynamics for i change. Experience goes stale.
#
# (B) Credit assignment (multi-agent)
#     Team got +10 — was it because of my move or yours? Harder than single-agent
#     temporal credit assignment.
#
# (C) Scalability of joint action spaces
#     |A| = |A_1| · |A_2| · … grows exponentially. Joint Q(s, a_1, a_2, …)
#     is only feasible for tiny discrete spaces (our "centralized Q" baseline).
#
# (D) Partial observability
#     Often each agent sees only a local view. Need policies over observations,
#     sometimes with memory (RNNs) — we keep Gridworld fully observed for clarity.
#
# (E) Equilibrium / which solution?
#     In competitive or mixed games there may be many Nash equilibria.
#     Cooperative MARL usually sidesteps this by optimizing a shared reward.
#
#
# ---------------------------------------------------------------------------
# 3) PARADIGMS (how people structure learning)
# ---------------------------------------------------------------------------
#
# Think of three axes people mix-and-match:
#
#   1. Independent vs centralized critics / training
#   2. Cooperative vs competitive vs mixed rewards
#   3. Tabular vs deep function approximation
#
# Common named paradigms:
#
#   Independent Learning (IL)
#     Each agent pretends others are part of the environment.
#     Train Q_i or π_i with ordinary single-agent algorithms (IQL, IPPO, …).
#     Simple and scalable; ignores non-stationarity formally, often still works
#     as a strong baseline.
#
#   Centralized Training, Decentralized Execution (CTDE)
#     During TRAINING, a critic (or planner) may see everyone's info.
#     At EXECUTION, each agent acts from its own observation only.
#     Examples: MADDPG, QMIX, MAPPO. Best of both worlds for many teams.
#
#   Fully Centralized
#     One brain picks the joint action. Optimal in tiny domains; does not scale.
#     Our "Centralized Q-Learning" is this — a teaching upper bound.
#
#   Value factorization (cooperative deep MARL)
#     Learn individual Q_i that sum / mix into a joint Q (VDN, QMIX).
#     Not implemented today; know the name for later reading.
#
#
# ---------------------------------------------------------------------------
# 4) WHAT WE IMPLEMENT (cooperative 2-agent Gridworld)
# ---------------------------------------------------------------------------
#
# Environment sketch (details in later cells):
#   Two agents on a small grid, shared or team reward for reaching goals /
#   coordinating. Fully observed states so tabular methods are honest.
#
#   ┌────────────────────────────────────────────────────────────┐
#   │  IQL   Independent Q-Learning (tabular)                    │
#   │        Each agent keeps its own Q_i(s, a_i).               │
#   │        Update like ordinary Q-learning, treating the other │
#   │        agent as "noise" in the environment.                │
#   │        = simple baseline; can be unstable but often OK.    │
#   ├────────────────────────────────────────────────────────────┤
#   │  Centralized Q  Joint tabular Q(s, a_1, a_2)               │
#   │        One table over joint actions.                       │
#   │        Near-optimal reference when |A|² · |S| still small. │
#   │        = upper bound / "what coordination can achieve".    │
#   ├────────────────────────────────────────────────────────────┤
#   │  IPPO  Independent PPO (neural)                            │
#   │        Each agent runs its own PPO actor–critic (Day 25).  │
#   │        Scales beyond tiny grids; modern default baseline.  │
#   └────────────────────────────────────────────────────────────┘
#
# How to read results later:
#   Centralized Q  ≳  IPPO  ≳  IQL     (typical on a tiny cooperative grid)
#   If IQL ≈ centralized, the task barely needs coordination.
#   If centralized >> IQL, coordination matters — IPPO should close some gap.
#
#
# ---------------------------------------------------------------------------
# BIG PICTURE
# ---------------------------------------------------------------------------
#
#   MDP (1 agent)  →  Markov Game (N agents)
#         │
#         ├─ Independent learners (IQL, IPPO)     ← scalable baselines
#         ├─ Centralized joint Q                  ← tiny-domain upper bound
#         └─ CTDE / value factorization           ← production cooperative MARL
#
# Next cells: build the Gridworld, then IQL → Centralized Q → IPPO.


In [ ]:
# Cooperative Two-Agent Gridworld
# ===============================
#
# TOPIC — the Markov Game we will train on
# ----------------------------------------
# Two agents share a size×size grid. Each picks a move every step.
# They succeed ONLY if BOTH sit on their own goals at the same time:
#
#   Agent 0: start top-left  (0,0)           → goal bottom-right (n-1, n-1)
#   Agent 1: start top-right (0, n-1)        → goal bottom-left  (n-1, 0)
#
#   Reward (team / cooperative):
#     +10  when both on goals together  → episode ends
#     −0.1 every other step             → finish fast
#     timeout at max_steps              → done with no bonus
#
# Why this env for MARL baselines?
#   - Fully observed joint state s = (pos0, pos1)  → tabular Q is honest
#   - Shared reward R0 = R1                        → pure cooperation
#   - Needs a bit of coordination (both must arrive; dawdling costs −0.1)
#   - Small discrete actions {up, down, left, right} each
#
# Formal map (Day 26 concepts):
#   S = all pairs of cell positions
#   A_i = {0,1,2,3}  (indices into self.actions)
#   joint action a = (a0, a1)
#   P = deterministic moves (walls = stay put)
#   R_i = team reward above
#
#
# CODE APPROACH
# -------------
# Setup cell: seeds + torch/numpy imports used by IQL / Centralized Q / IPPO later.
# CoopGridWorld:
#   reset() → joint state tuple
#   step([a0, a1]) → (next_state, [r, r], done)   # same r for both agents


import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from collections import defaultdict
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class CoopGridWorld:
    """Fully cooperative 2-agent grid: both must reach their goals together."""

    def __init__(self, size=5, max_steps=50):
        self.size = size
        self.max_steps = max_steps
        self.actions = ["up", "down", "left", "right"]
        self.action_effects = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1),
        }
        # Opposite corners: forces agents to cross the board
        self.start = [(0, 0), (0, self.size - 1)]
        self.goals = [(self.size - 1, self.size - 1), (self.size - 1, 0)]
        self.gamma = 0.95
        self.step_reward = -0.1      # cost of time
        self.success_reward = 10.0  # team bonus when both on goals
        self.reset()

    def reset(self):
        self.positions = list(self.start)  # mutable copy of start cells
        self.t = 0
        self.done = False
        return tuple(self.positions)  # joint state s = (pos0, pos1)

    def _move(self, pos, action):
        # Attempt one step; if it would leave the board, stay (bounce off wall)
        di, dj = self.action_effects[action]
        ni, nj = pos[0] + di, pos[1] + dj
        if 0 <= ni < self.size and 0 <= nj < self.size:
            return (ni, nj)
        return pos

    def step(self, actions):
        # actions: [a0, a1] with each a_i an index into self.actions
        assert not self.done
        self.positions[0] = self._move(self.positions[0], self.actions[actions[0]])
        self.positions[1] = self._move(self.positions[1], self.actions[actions[1]])
        self.t += 1

        both_on_goal = (
            self.positions[0] == self.goals[0]
            and self.positions[1] == self.goals[1]
        )
        if both_on_goal:
            reward = self.success_reward
            self.done = True
        else:
            reward = self.step_reward
            if self.t >= self.max_steps:
                self.done = True  # timeout failure (no +10)

        # Same reward for both agents → cooperative Markov game
        return tuple(self.positions), [reward, reward], self.done

# Quick sanity check:
env = CoopGridWorld()
s = env.reset()
print("Initial joint state:", s)
s, r, d = env.step([1, 1])  # both move down
print("After both move down:", s, r, d)

In [ ]:
# Independent Q-Learning (IQL)
# ============================
# TOPIC — each agent learns as if it were alone
# ---------------------------------------------
# IQL = ordinary Q-learning, run once per agent, on the SHARED joint state s.
#
#   Agent i stores Q_i(s, a_i)   ← only ITS action, not the teammate's
#   Update (same as single-agent Q-learning):
#     Q_i(s, a_i) ← Q_i + α [ r + γ max_{a'} Q_i(s', a') − Q_i(s, a_i) ]
#
# The other agent is treated as "part of the environment." That environment
# is NON-STATIONARY (teammate is also learning) — IQL ignores that, which is
# why it is a simple baseline, not a perfect MARL solution.
#
# Exploration: ε-greedy per agent, ε decays over episodes.
# Success metric: fraction of episodes where both finish on their goals.


def iql_train(
    env,
    num_episodes=4000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=500,
):
    # One Q-table per agent: key = joint state s, value = array of 4 action values
    Q = [defaultdict(lambda: np.zeros(4)), defaultdict(lambda: np.zeros(4))]
    success_rates = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        steps = 0
        while not done:
            # --- Act: each agent ε-greedy on its own Q_i ---
            acts = []
            for i in range(2):
                if random.random() < epsilon:
                    acts.append(random.randrange(4))  # explore
                else:
                    acts.append(int(np.argmax(Q[i][s])))  # exploit

            s_next, rewards, done = env.step(acts)
            r = rewards[0]  # team reward (same for both)

            # --- Learn: independent TD updates (teammate action not in the table) ---
            for i in range(2):
                best_next = 0.0 if done else np.max(Q[i][s_next])
                td_target = r + gamma * best_next
                Q[i][s][acts[i]] += alpha * (td_target - Q[i][s][acts[i]])

            s = s_next
            steps += 1

        # Decay exploration over training
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        # Did this episode succeed (both on goals when done)?
        success = (
            env.positions[0] == env.goals[0]
            and env.positions[1] == env.goals[1]
        )
        success_rates.append(1 if success else 0)

        if (ep + 1) % print_every == 0:
            rate = np.mean(success_rates[-print_every:])
            print(
                f"IQL Ep {ep+1} | Success rate (last {print_every}): {rate:.2f} "
                f"| eps={epsilon:.3f}"
            )

    return Q, success_rates


In [ ]:
# Centralized Q-Learning (Joint Action)
# =====================================
#
# CONCEPT — fully centralized MARL (tiny-domain upper bound)
# ----------------------------------------------------------
# IQL (previous cell): each agent learns Q_i(s, a_i) and ignores the teammate's
# choice inside its table. Coordination must emerge accidentally.
#
# Centralized Q: treat the TEAM as ONE super-agent whose action is the pair
# (a0, a1). Learn a single table:
#
#   Q(s, a0, a1)   ≈   Q(s, ja)   with ja ∈ {0,…,15}
#
# Encoding (4 actions each → 4² = 16 joint actions):
#   ja = 4*a0 + a1
#   a0 = ja // 4 ,  a1 = ja % 4
#
# Why this matters:
#   + Can represent ANY coordinated strategy (who waits, who goes first, …)
#   + Same math as ordinary Q-learning → easy teaching reference
#   − Joint action space grows as |A|^N  → only OK for small discrete teams
#
# In Day 26 paradigms this is "fully centralized control." Use it as an
# UPPER BOUND: if IQL << Centralized Q, the task needs real coordination.
#
#
# CODE APPROACH
# -------------
# One defaultdict Q[s] → length-16 array
# Loop: ε-greedy over ja → decode (a0,a1) → env.step → TD update on ja
# Track success rate like IQL for fair comparison plots later.


def centralized_q_train(
    env,
    num_episodes=2000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=250,
):
    # Joint Q-table: 16 values = every (a0, a1) pair
    Q = defaultdict(lambda: np.zeros(16))
    success_rates = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        while not done:
            # Explore / exploit in JOINT action space (not per-agent)
            if random.random() < epsilon:
                ja = random.randrange(16)
            else:
                ja = int(np.argmax(Q[s]))

            # Unpack joint index into the two agents' moves
            a0, a1 = ja // 4, ja % 4
            s_next, rewards, done = env.step([a0, a1])
            r = rewards[0]  # shared team reward

            # Classic Q-learning: bootstrap with max over next joint actions
            best_next = 0.0 if done else np.max(Q[s_next])
            td_target = r + gamma * best_next
            Q[s][ja] += alpha * (td_target - Q[s][ja])
            s = s_next

        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        # Success = both agents on their goals when the episode ended
        success = (
            env.positions[0] == env.goals[0]
            and env.positions[1] == env.goals[1]
        )
        success_rates.append(1 if success else 0)

        if (ep + 1) % print_every == 0:
            rate = np.mean(success_rates[-print_every:])
            print(f"CQL Ep {ep+1} | Success rate: {rate:.2f} | eps={epsilon:.3f}")

    return Q, success_rates


In [ ]:
# Independent PPO (IPPO)
# ======================
#
# CONCEPT — scalable neural baseline for MARL
# -------------------------------------------
# IPPO = each agent runs its OWN PPO actor–critic (Day 25), independently.
#
#   Agent i:  π_i(a_i | s)  and  V_i(s)
#   Same joint state s for both (fully observed Gridworld)
#   Shared team reward r, but SEPARATE networks / optimizers
#
# Like IQL, teammates are treated as part of the environment (non-stationary).
# Unlike IQL, policies are neural nets → scales beyond tiny tabular grids.
#
# vs other baselines in this notebook:
#   IQL            — tabular independent Q (simple, limited)
#   Centralized Q  — joint action table (upper bound, tiny only)
#   IPPO           — deep independent policy gradient + clip + GAE
#
# This is "independent learning" in the Day 26 paradigms — NOT CTDE
# (no centralized critic that sees both agents' internals).
#
#
# CODE MAP
# --------
# ActorCritic     — shared trunk → actor logits + critic V  (per agent)
# state_to_vec    — one-hot both positions → fixed-size input vector
# compute_gae     — advantages for PPO (same idea as Day 25)
# ippo_train      — collect joint rollouts → PPO update each agent


class ActorCritic(nn.Module):
    """One agent's policy π(a|s) + value V(s). Instantiated twice for IPPO."""

    def __init__(self, state_dim, n_actions, hidden_dim=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)
        # Orthogonal init (PPO-stable defaults)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)  # start near-uniform π
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h).squeeze(-1)


def state_to_vec(joint_state, size=5):
    """Encode s = (pos0, pos1) as concatenated one-hots → length 2*size*size."""
    vec = np.zeros(2 * size * size, dtype=np.float32)
    for i, (r, c) in enumerate(joint_state):
        vec[i * size * size + r * size + c] = 1.0
    return vec


def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """Backward GAE: Â_t = δ_t + γλ(1−done) Â_{t+1} ; returns = Â + V."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last = 0.0
    for t in reversed(range(T)):
        nv = next_value if t == T - 1 else values[t + 1]
        nt = 1.0 - dones[t]
        delta = rewards[t] + gamma * nv * nt - values[t]
        last = delta + gamma * lam * nt * last
        adv[t] = last
    returns = adv + values
    return adv, returns


def ippo_train(
    env,
    num_updates=200,
    rollout_len=512,
    gamma=0.95,
    lam=0.95,
    clip_eps=0.2,
    value_coef=0.5,
    entropy_coef=0.01,
    ppo_epochs=6,
    lr=3e-4,
    print_every=20,
):
    state_dim = 2 * env.size * env.size
    # Independent learners: separate nets + Adam for each agent
    policies = [ActorCritic(state_dim, 4).to(device) for _ in range(2)]
    optimizers = [optim.Adam(p.parameters(), lr=lr) for p in policies]

    success_rates = []

    for update in range(num_updates):
        # --- 1) Collect a joint rollout (both agents act every step) ---
        bufs = [[] for _ in range(2)]  # per-agent transition lists
        s = env.reset()
        ep_successes = []
        steps = 0
        while steps < rollout_len:
            sv = state_to_vec(s, env.size)
            sv_t = torch.FloatTensor(sv).unsqueeze(0).to(device)
            acts, logps, vals = [], [], []
            for i in range(2):
                with torch.no_grad():
                    logits, v = policies[i](sv_t)
                    dist = Categorical(logits=logits)
                    a = dist.sample()
                    acts.append(a.item())
                    logps.append(dist.log_prob(a).item())  # π_old for ratio
                    vals.append(v.item())
            s_next, rewards, done = env.step(acts)
            r = rewards[0]  # team reward copied into both buffers
            for i in range(2):
                bufs[i].append((sv, acts[i], logps[i], r, vals[i], float(done)))
            s = s_next
            steps += 1
            if done:
                success = (
                    env.positions[0] == env.goals[0]
                    and env.positions[1] == env.goals[1]
                )
                ep_successes.append(1 if success else 0)
                s = env.reset()

        # --- 2) PPO update EACH agent on its own buffer (independent) ---
        for i in range(2):
            states = torch.tensor(
                np.stack([b[0] for b in bufs[i]]), dtype=torch.float32, device=device
            )
            actions = torch.tensor(
                [b[1] for b in bufs[i]], dtype=torch.long, device=device
            )
            old_logp = torch.tensor(
                [b[2] for b in bufs[i]], dtype=torch.float32, device=device
            )
            rewards = [b[3] for b in bufs[i]]
            values = [b[4] for b in bufs[i]]
            dones = [b[5] for b in bufs[i]]

            # Bootstrap V(s_T) for GAE at end of fragment
            with torch.no_grad():
                _, nv = policies[i](
                    torch.tensor(
                        state_to_vec(s, env.size), dtype=torch.float32, device=device
                    ).unsqueeze(0)
                )
                next_v = nv.item()

            adv, ret = compute_gae(rewards, values, dones, next_v, gamma, lam)
            adv = torch.tensor(adv, dtype=torch.float32, device=device)
            ret = torch.tensor(ret, dtype=torch.float32, device=device)
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)

            # Multi-epoch clipped PPO (same loss as Day 25)
            for _ in range(ppo_epochs):
                logits, vs = policies[i](states)
                dist = Categorical(logits=logits)
                new_logp = dist.log_prob(actions)
                ratio = torch.exp(new_logp - old_logp)
                s1 = ratio * adv
                s2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
                ploss = -torch.min(s1, s2).mean()
                vloss = F.mse_loss(vs, ret)
                ent = dist.entropy().mean()
                loss = ploss + value_coef * vloss - entropy_coef * ent
                optimizers[i].zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policies[i].parameters(), 0.5)
                optimizers[i].step()

        if ep_successes:
            success_rates.extend(ep_successes)
        if (update + 1) % print_every == 0:
            recent = success_rates[-200:]
            print(
                f"IPPO Update {update+1} | "
                f"Success rate (last {len(recent)}): {np.mean(recent):.2f}"
            )

    return policies, success_rates

# ---------------------------------------------------------------------------
# Train and Compare — run all three MARL baselines on fresh Gridworlds
# ---------------------------------------------------------------------------
# Fresh env per method so episode counters / positions don't leak across runs.
# Each returns a success_rates list (1/0 per episode) for later plotting.
#
# Expected rough ordering on this cooperative task:
#   Centralized Q  ≳  IPPO  ≳  IQL
# (joint Q can coordinate; IPPO scales; IQL is the simplest independent baseline)
#
# Episode/update counts are demo-sized — raise them for smoother curves.

# IQL: tabular independent Q-learning (~3000 episodes)
env = CoopGridWorld()
Q_iql, sr_iql = iql_train(env, num_episodes=3000)
print()

# Centralized Q: joint-action table (fewer episodes often enough — denser updates)
env = CoopGridWorld()
Q_cql, sr_cql = centralized_q_train(env, num_episodes=1500)
print()

# IPPO: two neural PPO agents, 150 updates × 512 rollout steps each
env = CoopGridWorld()
policies_ippo, sr_ippo = ippo_train(env, num_updates=150, rollout_len=512)


# Plot smoothed success rates side‑by‑side:
def smooth(x, w=100):
    if len(x) < w: return x
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smooth(sr_iql), label='IQL')
plt.plot(smooth(sr_cql), label='Centralized Q')
plt.plot(smooth(sr_ippo), label='IPPO')
plt.xlabel('Episode / update (rolling window)')
plt.ylabel('Success rate')
plt.title('Cooperative Two-Agent Gridworld — MARL Baselines')
plt.legend()
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# Hands‑On Exercises / Mini‑Project
# 3.1 Coordinated vs. Independent Start
# Modify the environment so that both agents start at (0,0) and must reach (4,4) and (4,0) respectively. Does IQL succeed? Where does it break down? Centralized Q should still solve it. This demonstrates credit assignment and coordination.

# 3.2 Communication Ablation (Conceptual)
# Add a communication channel: at each step, each agent emits a discrete message in {0,1,2,3} that the other sees. This is a very simple version of commNet. Train with IQL or IPPO. Does the success rate improve, especially with the coordinated start?

# 3.3 Competitive Variant
# Change the reward so that +1 is given to whichever agent reaches its own goal first, and 0 otherwise (agents become competitors). Observe how IQL and IPPO produce competitive behavior (agents racing to their goals). This is a mixed/competitive setting that introduces the idea of equilibrium.

# 3.4 PettingZoo Cross‑Check
# If you have PettingZoo installed, replicate IPPO on pettingzoo.mpe.simple_spread_v3 (a cooperative particle task). This is a standard MARL benchmark and makes a strong portfolio comparison.

# python
# # !pip install pettingzoo
# # from pettingzoo.mpe import simple_spread_v3
# 3.5 Portfolio Artifact
# Create day26_marl_baselines.ipynb with:

# Environment description and screenshots (or ASCII diagrams of the grid at various steps).

# IQL, Centralized Q, and IPPO implementations.

# Learning‑curve comparison.

# Ablations (coordinated start, communication).

# Discussion of non‑stationarity, credit assignment, and CTDE.

# Save trained IPPO policies:

# python
# torch.save(policies_ippo[0].state_dict(), 'ippo_agent0.pt')
# torch.save(policies_ippo[1].state_dict(), 'ippo_agent1.pt')
